In [ ]:
import fastf1
import pandas as pd
import os

In [ ]:
# # sign into f1 account
# os.environ['F1_USERNAME'] = 'username'
# os.environ['F1_PASSWORD'] = 'password'


schedule_df = pd.read_csv('./events/events.csv')
sessions = ['Session2', 'Session3', 'Session4', 'Session5']
year = 2021
races = ['Russian Grand Prix', 'Turkish Grand Prix', 'Qatar Grand Prix', 'Saudi Arabian Grand Prix'] 
failed_races = ''
for race in races:
    race_df = schedule_df.loc[(schedule_df['EventName'] == race) & (schedule_df['Year'] == year)].reset_index(drop=True)
    for session in sessions:
        session_type = race_df[session].iloc[0]
        if session_type in ['Race', 'Sprint', 'Qualifying', 'Sprint Qualifying', 'Sprint Shootout']:
            try:
                session_data = fastf1.get_session(year, race, session_type)
                session_data.load()

                session_type = session_data.session_info['Type']
                name = f'{year}_{race}_{session_type}'
                results = session_data.results
                laps = session_data.laps
                weather = session_data.weather_data

                # exporting dfs to parquets

                results.to_parquet(f'./data/{name}_results.parquet'.lower().replace(' ', '').replace(' ', ''), engine='pyarrow', compression='snappy')
                laps.to_parquet(f'./data/{name}_laps.parquet'.lower().replace(' ', '').replace(' ', ''), engine='pyarrow', compression='snappy')
                weather.to_parquet(f'./data/{name}_weather.parquet'.lower().replace(' ', '').replace(' ', ''), engine='pyarrow', compression='snappy')

            except:
                failed_races = failed_races + f'<{race} - {session_type} Failed\n'


failed_races


In [ ]:
# error testing
from f1_analyzer_2 import F1Analysis as f1

events = pd.read_csv('./events/events.csv')
events = events.loc[events['Complete'] == True].copy()
valid_sessions = ['Race', 'Sprint', 'Qualifying', 'Sprint Qualifying', 'Sprint Shootout']
session_labels = ['Session2', 'Session3', 'Session4', 'Session5']

error_str = ''
for year in list(range(2019, 2026 + 1)):
    year_df = events.loc[events['Year'] == year].reset_index(drop=True).copy()
    for track in list(year_df['EventName']):
        track_df = year_df.loc[year_df['EventName'] == track].reset_index(drop=True).copy()
        v_sessions = []
        for s in session_labels:
            ses = track_df[s].iloc[0]
            if ses in valid_sessions:
                v_sessions.append(ses)
        for v in v_sessions:
            try:
                x = f1(year,track,v)
            except:
                error_str += f'{year}_{track}_{v}\n'.strip().lower()

error_str
        


In [ ]:
# scraping wiki for missing data
# lap times missing in 2025 miami quali results
from bs4 import BeautifulSoup as bs
import requests

url = 'https://en.wikipedia.org/wiki/2025_Miami_Grand_Prix'



fake_header = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Connection": "keep-alive"
}

scraped_dfs = []


req = requests.get(url, headers=fake_header)
soup = bs(req.text, 'html.parser')

table = soup.find_all('table')[4]
cols = [x.text.strip() for x in table.find_all('th')]
rows = [x.text.strip() for x in table.find_all('td')][:-2]
cols.append('Grid')
cols.remove('Qualifying times')
cols.remove('Finalgrid')
x = 1
lab = {}
for c in cols:
    lab[x] = c
    x += 1

x = 1
data = {c: [] for c in cols}

for r in rows:
    data[lab[x]].append(r)
    x += 1
    if x > len(cols ):
        x = 1

df = pd.DataFrame(data)

# converting string lap times to numbers
def lap_time_to_numeric(lap):
    if len(lap) != 8:
        return pd.NA
    mins = int(lap[0]) * 60
    secs = int(lap[2:4])
    mill = float(lap[4:])

    return mins + secs + mill

for q in ['Q1', 'Q2', 'Q3']:
    df[q] = df[q].apply(lap_time_to_numeric)

# updating incomplete data
path = './data/2025_miamigrandprix_qualifying_results.parquet'
parq = pd.read_parquet(path)

for q in ['Q1', 'Q2', 'Q3']:
    parq[q] = list(df[q])
    parq[q] = pd.to_timedelta(parq[q], unit='s') #raw times come in timedelta and converted to seconds later

parq.to_parquet(path, compression='snappy', engine='pyarrow')

In [ ]:
# broken tyre data fix for qatar sprint 2024, and miami gp 25
path = './data/2023_qatargrandprix_sprint_laps.parquet'
df = pd.read_parquet(path)

good = []

for driver in list(df['Driver'].unique()):
    df_p = df.loc[df['Driver'] == driver]

    if 'nan' not in list(df_p['Compound']):
        good.append(df_p)
        continue

    df_no_n = df_p.loc[df['Compound'] != 'nan']
    stint = df_no_n['Stint'].iloc[0]
    comp = df_no_n['Compound'].iloc[0]

    df_p.loc[df_p['Stint'].isnull(), 'Stint'] = stint
    df_p.loc[df_p['Compound'] == 'nan', 'Compound'] = comp
    df_p.loc[df_p['Stint'] == stint, 'TyreLife'] = list(range(1, len(list(df_p.loc[df_p['Stint'] == stint].index)) + 1))

    good.append(df_p)


new_df = pd.concat(good, ignore_index=True)
new_df.to_parquet(path, engine='pyarrow', compression='snappy')